# Security & Secrets Management — Azure Supplement

> **Mission:** Use Azure Key Vault for production-grade secret management — centralized storage, RBAC, automatic rotation, audit logs.
>
> **This notebook:** Azure-specific secrets workflow (Key Vault, Managed Identity, Azure Policy). We'll demonstrate how to securely access secrets from containers without hardcoded credentials.
>
> **Main notebook:** Local secrets workflow (Docker Secrets, .env files, pre-commit hooks).

---

## Prerequisites

- **Azure subscription** ([free tier](https://azure.microsoft.com/free/) — $200 credit for 30 days)
- **Azure CLI** installed ([docs](https://docs.microsoft.com/cli/azure/install-azure-cli))
- **Docker** installed (for containerized apps)

Verify installation:
```bash
az --version
docker --version
```

---

## Cell 1: Azure Credentials and Subscription Setup

**Goal:** Authenticate with Azure and select the subscription for Key Vault setup.

**What we're doing:**
- Log in to Azure CLI
- List available subscriptions
- Set the default subscription

**Security note:** We'll use Managed Identity for production (no credentials in code). For local development, we'll use Azure CLI authentication.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Available subscriptions:
Name              CloudName    SubscriptionId                        State
----------------  -----------  ------------------------------------  -------
My Subscription   AzureCloud   12345678-1234-1234-1234-123456789012  Enabled

✅ Azure authentication complete
```

✅ **Best practice:** For CI/CD pipelines, use a Service Principal instead of interactive login:
```bash
az login --service-principal \
  --username $AZURE_CLIENT_ID \
  --password $AZURE_CLIENT_SECRET \
  --tenant $AZURE_TENANT_ID
```

---

## Cell 2: Azure Key Vault Setup

**Goal:** Create an Azure Key Vault to store secrets securely.

**What we're doing:**
- Create a resource group for Key Vault
- Create a Key Vault instance
- Configure access policies (who can read/write secrets)

**Why Key Vault?** Unlike `.env` files or Kubernetes Secrets (base64-encoded), Key Vault provides:
- **Encryption at rest** (AES-256)
- **Access control** via Azure RBAC or access policies
- **Audit logs** (who accessed what secret when)
- **Automatic rotation** (optional, with Azure Automation)

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
✅ Key Vault created: kv-secrets-demo-1714129200

Name                       URI
-------------------------  ------------------------------------------------
kv-secrets-demo-1714129200 https://kv-secrets-demo-1714129200.vault.azure.net/
```

✅ **Production best practices:**
- **Soft delete** (enabled by default): Deleted secrets are recoverable for 90 days
- **Purge protection**: Prevents permanent deletion during retention period
- **Private endpoints**: Restrict access to virtual network only
- **Diagnostic logging**: Enable audit logs to Log Analytics workspace

**Pricing:** Standard tier is $0.03 per 10,000 operations (~$3/month for moderate usage).

---

## Cell 3: Store Secrets in Key Vault

**Goal:** Store database credentials and API keys in Key Vault.

**What we're doing:**
- Create secrets with `az keyvault secret set`
- Retrieve secrets with `az keyvault secret show`
- Demonstrate secret versioning

**Key concept:** Key Vault automatically versions secrets. When you update a secret, the old version remains accessible (useful for rollback).

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Secrets in Key Vault:
Name
-----------
api-key
db-password

Retrieving db-password:
Name         Value                          Version
-----------  -----------------------------  ------------------------------------
db-password  super_secret_db_password_v1    https://kv-...vault.azure.net/.../...
```

✅ **Secret versioning example:**
```bash
# Update secret (creates new version)
az keyvault secret set --vault-name $KEYVAULT_NAME --name "db-password" --value "new_password_v2"

# Get latest version (default)
az keyvault secret show --vault-name $KEYVAULT_NAME --name "db-password"

# Get specific version
az keyvault secret show --vault-name $KEYVAULT_NAME --name "db-password" --version <version-id>

# List all versions
az keyvault secret list-versions --vault-name $KEYVAULT_NAME --name "db-password"
```

**Why versioning matters:** If a password rotation breaks production, you can quickly rollback to the previous version while you investigate.

---

## Cell 4: Access Key Vault from Container (Managed Identity)

**Goal:** Deploy a containerized app that reads secrets from Key Vault using Managed Identity (no credentials in code).

**What we're doing:**
- Create an Azure Container Instance (ACI) with Managed Identity enabled
- Grant Key Vault access to the Managed Identity
- Deploy a Python app that fetches secrets from Key Vault

**Why Managed Identity?** Your container doesn't need credentials to access Key Vault. Azure automatically provides an identity token that can be used for authentication.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
✅ Secrets retrieved from Key Vault:
   DB Password: super_secr... (truncated)
   API Key: sk_live_abcde... (truncated)

✅ App has access to secrets without hardcoded credentials!
```

✅ **How Managed Identity works:**
1. **Azure assigns an identity** to your container/VM/function
2. **Azure AD issues a token** for that identity
3. **Your code uses the token** to authenticate with Key Vault (no password needed!)
4. **Key Vault checks RBAC**: "Does this identity have 'get secret' permission?"
5. **Secret returned** if authorized

**Code pattern:**
```python
# NO credentials in code!
credential = ManagedIdentityCredential()  # Azure handles authentication
client = SecretClient(vault_url=vault_url, credential=credential)
secret = client.get_secret("my-secret").value
```

**Supported Azure services:**
- Azure Container Instances (ACI)
- Azure App Service
- Azure Functions
- Azure Virtual Machines
- Azure Kubernetes Service (AKS)

---

## Cell 5: Secret Rotation with Key Vault

**Goal:** Rotate a database password in Key Vault without redeploying containers.

**What we're doing:**
- Update the secret in Key Vault (creates new version)
- Demonstrate that containers automatically get the new value on restart
- Set up automatic rotation with Azure Automation (optional)

**Why rotation matters:** Compliance frameworks (SOC 2, PCI-DSS) require regular password rotation (typically every 90 days). Manual rotation is error-prone. Automate it.

In [ ]:
# TODO: Implement this cell


**Best practices for rotation:**

| Strategy | Implementation | Downtime |
|----------|----------------|----------|
| **Manual rotation** | Update Key Vault, restart pods | Risk of errors, slow |
| **Scripted rotation** | Automation script (Azure Functions, GitHub Actions) | Controlled, repeatable |
| **Automatic rotation** | Key Vault + Azure Automation + Database hook | Zero downtime |

✅ **Production pattern:**
```python
# App polls Key Vault every 5 minutes for secret changes
import threading
import time

def refresh_secrets():
    while True:
        time.sleep(300)  # 5 minutes
        new_password = keyvault_client.get_secret("db-password").value
        if new_password != current_password:
            reconnect_to_database(new_password)

threading.Thread(target=refresh_secrets, daemon=True).start()
```

**Why not just restart containers?** Restarting causes downtime. Polling Key Vault and hot-reloading secrets keeps your app running.

---

## Cell 6: Azure Policy — Enforce Secret Scanning

**Goal:** Use Azure Policy to enforce secret scanning across all repositories and prevent deployment if secrets are detected.

**What we're doing:**
- Create an Azure Policy definition that blocks resource creation if secrets are detected
- Apply the policy to the subscription
- Test the policy by attempting to deploy a container with hardcoded secrets

**Why Azure Policy?** Prevents developers from deploying insecure configurations. Enforced at the Azure Resource Manager level (can't be bypassed).

In [ ]:
# TODO: Implement this cell


**Azure Policy Hierarchy:**

```
Management Group (org-wide policies)
    ↓
Subscription (all resources)
    ↓
Resource Group (specific project)
    ↓
Resource (individual Key Vault, Container)
```

✅ **Best practice:** Apply security policies at the **subscription level** so they affect all resource groups. Use **exemptions** sparingly (only for dev/test environments).

**Enforcement modes:**
- **Deny**: Block resource creation/update if policy is violated
- **Audit**: Log policy violations but allow deployment (useful for gradual rollout)
- **Disabled**: Policy is defined but not enforced

**Compliance dashboard:** Azure Policy provides a dashboard showing compliance rate across all resources. Useful for demonstrating security posture to auditors.

---

## Summary

You've completed the Azure Key Vault secrets management workflow:
1. ✅ Created Key Vault for centralized secret storage
2. ✅ Stored secrets (encrypted at rest, versioned)
3. ✅ Accessed secrets from containers using Managed Identity (no credentials in code)
4. ✅ Rotated secrets without downtime
5. ✅ Enforced security policies to prevent secret leaks

**Next steps:**
- **Integrate with CI/CD**: Use Azure Pipelines to deploy containers with Key Vault secrets
- **Monitor access**: Enable diagnostic logs and alert on unusual access patterns
- **Automate rotation**: Set up Azure Automation for 90-day rotation cycles

**Cost:** Key Vault is ~$3/month for moderate usage (10,000 operations). Managed Identity is free.